# 03. Верификация латинских моделей

    Эти проверки **не влияют на выбор представительной модели**.

    1. Тест синонимов Спрунцоли, Пассаротти и Моретти, сравнение ранней LatinISE с *Opera Latina* на общих заданиях;
    2. 30 диагностических пар слов: средний косинус и разброс по десяти запускам;
    3. Сопоставление соседей *uirtus* и *dominus* с опубликованными данными Лунарди.

    Тест синонимов ограничен охватом словаря, особенно при `min_count=50`. Пары — примеры, а не экспертный эталон.

In [1]:
from pathlib import Path
import sys

# При необходимости укажите каталог на Google Drive:
PROJECT_ROOT = None  # например: '/content/drive/MyDrive/latin-embeddings-models'

candidates = [Path(PROJECT_ROOT)] if PROJECT_ROOT else [Path.cwd(), Path.cwd().parent, Path.cwd() / 'latin-embeddings-models']
root = next((p.resolve() for p in candidates if (p / 'src/core.py').exists()), None)
if root is None:
    raise FileNotFoundError('Укажите PROJECT_ROOT — папку репозитория.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Проект:', root)

Проект: C:\Users\Admin\Documents\GitHub\latinise-humanities


In [2]:
from src import core, validation as vt
from IPython.display import display
_, selected = core.load_selected()
out = core.RESULTS / 'validation'
out.mkdir(parents=True, exist_ok=True)

## 1. Стабильность и диагностические пары

In [3]:
display(vt.neighbor_reproducibility(out))
display(vt.diagnostic_pairs(selected, out).head(12))

,slice,min_count,mean_anchor_jaccard,median_anchor_jaccard,n_anchors
0,pre180,5,0.498006,0.497347,300
1,post180,5,0.517548,0.510852,300
2,pre180,50,0.565319,0.570968,300
3,post180,50,0.577763,0.570185,300


,slice,min_count,word1,word2,relation_type,n_runs,mean_cosine,sd_cosine
0,pre180,5,rex,regnum,institution,10,0.522925,0.009354
1,pre180,5,consul,senatus,institution,10,0.455195,0.018124
2,pre180,5,imperator,imperium,institution,10,0.418555,0.033295
3,pre180,5,miles,exercitus,topic,10,0.602901,0.010008
4,pre180,5,hostis,bellum,topic,10,0.142532,0.014337
5,pre180,5,equus,eques,topic,10,0.334708,0.012473
6,pre180,5,nauis,mare,topic,10,0.456145,0.028744
7,pre180,5,gladius,ferrum,topic,10,0.719148,0.013600
8,pre180,5,urbs,oppidum,relatedness,10,0.741215,0.012545
9,pre180,5,pater,filius,relation,10,0.399512,0.022802


## 2. Сопоставление с Лунарди

In [4]:
display(vt.lunardi_reference(selected, out))

,slice,min_count,lemma,reference_window,reference_min_count,matching_hyperparameters,overlap_top10,reference_neighbors,our_neighbors,note
0,pre180,5,dominus,10,50,False,6,"seruus, uenalis, emptor, mancipium, uendo, pra...","seruus, uenalis, pauper, seruitus, mancipium, ...",qualitative reference only; corpus/case/tokeni...
1,pre180,5,uirtus,5,50,False,9,"sapientia, laus, integritas, fortitudo, pruden...","laus, sapientia, fortitudo, constantia, pruden...",qualitative reference only; corpus/case/tokeni...
2,pre180,50,dominus,10,50,False,7,"seruus, uenalis, emptor, mancipium, uendo, pra...","seruus, pauper, mancipium, uenalis, emptor, de...",qualitative reference only; corpus/case/tokeni...
3,pre180,50,uirtus,5,50,True,7,"sapientia, laus, integritas, fortitudo, pruden...","sapientia, prudentia, fortitudo, constantia, l...",qualitative reference only; corpus/case/tokeni...
4,post180,5,dominus,10,50,False,1,"seruus, libertas, deus, domina, famulus, domin...","deus, pharao, israel, hodie, glorifico, moses,...",qualitative reference only; corpus/case/tokeni...
5,post180,5,uirtus,5,50,False,8,"bonitas, potentia, fortitudo, gloria, laus, pr...","fortitudo, potentia, prudentia, bonitas, glori...",qualitative reference only; corpus/case/tokeni...
6,post180,50,dominus,10,50,False,1,"seruus, libertas, deus, domina, famulus, domin...","deus, pharao, iacob, moses, dauid, israel, glo...",qualitative reference only; corpus/case/tokeni...
7,post180,50,uirtus,5,50,True,7,"bonitas, potentia, fortitudo, gloria, laus, pr...","bonitas, fortitudo, potentia, prudentia, glori...",qualitative reference only; corpus/case/tokeni...


## 3. Тест синонимов и Opera Latina

    Файлы можно предварительно положить вручную в:

    - `data/benchmark/syn-selection-benchmark-Latin.tsv`;
    - `data/reference_models/allLASLAlemmi-vector-100-nocase-w5-CBOW.vec`.

    Если файлов нет, программа попробует скачать их автоматически.

In [5]:
benchmark = vt.ensure_benchmark()
opera = vt.load_opera(vt.ensure_opera())
tasks = vt.parse_benchmark(benchmark)
report = vt.synonym_reports(selected, opera, tasks, out)
display(report)

,model,evaluation,n_eligible,n_covered,coverage,accuracy
0,pre180_mc5,own_available,2759,1382,0.500906,0.811143
1,pre180_mc50,own_available,2759,43,0.015585,0.906977
2,post180_mc5,own_available,2759,1084,0.392896,0.785978
3,post180_mc50,own_available,2759,51,0.018485,0.764706
4,opera_latina,own_available,2759,1751,0.634650,0.811536
5,pre180_mc5,shared:pre180_mc5|opera_latina,1167,1167,1.000000,0.832905
6,opera_latina,shared:pre180_mc5|opera_latina,1167,1167,1.000000,0.809769
7,pre180_mc50,shared:pre180_mc50|opera_latina,42,42,1.000000,0.904762
8,opera_latina,shared:pre180_mc50|opera_latina,42,42,1.000000,0.880952
9,pre180_mc5,shared:pre180_mc5|pre180_mc50,43,43,1.000000,0.953488


**Как читать отчёт:** сравнение моделей допустимо на одинаковом наборе заданий (`shared`). Процент правильных ответов обязательно сопровождается числом доступных вопросов. Различия с опубликованными соседями Лунарди могут быть связаны с разными корпусами и условиями обработки текстов.